# 19. Explorar alrededor de 1500 árboles, paso 100

Parte de `data/competencia_01.csv`. Ejecutar en un kernel nuevo. Reconstruye lags/deltas y normalización por máximo absoluto; agrega tendencia y promedio de tres observaciones; ejecuta canaritos; busca hiperparámetros con cinco trials; valida diez semillas y reentrena con marzo–junio para predecir agosto.

No agrega flags cero. No lee la base Optuna, modelos, datasets FE ni selecciones de otras corridas. Usa el rango de árboles 1000–2000 en pasos de 100. Se mantienen parámetros fijos, sin undersampling y semilla 17 de búsqueda. Con cinco trials, TPE predeterminado realiza su fase inicial aleatoria; se conserva esa configuración para repetir la experimentación real.

Referencia: `SofiaHist3Ensamble_20261010_203913_top12500.csv`, resultado público 99,7975. La coincidencia no está verificada: se debe comparar el CSV generado contra el original. La lista anterior de 400 variables se incorpora únicamente para auditar la selección nueva, no para reemplazarla. Tampoco se fijan los hiperparámetros ganadores: salen de Optuna.

Colocar opcionalmente el CSV ganador en `referencia/SofiaHist3Ensamble_20261010_203913_top12500.csv` para comparar clientes, orden y bytes al final. Los nombres y fechas de los nuevos archivos serán distintos.

Experimento nuevo: único cambio en búsqueda respecto a la 18, rango y paso de árboles. Hojas (8–128) y Hessian (0,1–100, log) conservan sus rangos originales. No fija esos dos hiperparámetros alrededor del ganador. Con 5 trials no recorre todas las cantidades de árboles y el sampler permanece en su fase inicial aleatoria. No se espera reproducir el CSV ganador; las comparaciones finales son referencias para medir el cambio. Conservar la 18 como reproducción del candidato ya enviado.


In [ ]:
%pip install pandas==3.0.6 numpy==2.5.3 lightgbm==4.7.0 optuna==5.0.0 ipykernel==7.3.0 duckdb


In [ ]:
from pathlib import Path
from datetime import datetime
import json, hashlib, platform, time
import importlib.metadata as metadata
import pandas as pd
import numpy as np
import lightgbm as lgb
import optuna
from IPython.display import display

BASE = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / "data" / "competencia_01.csv").is_file()), Path.cwd())
CSV = BASE / "data" / "competencia_01_fe_lags_norm.csv"
CSV_ORIGINAL = BASE / "data" / "competencia_01.csv"
MESES_BUSQUEDA = [202103, 202104]
MES_VALIDACION = 202106
MESES_FINAL = [202103, 202104, 202105, 202106]
MES_PREDICCION = 202108
N_TRIALS = 5
OBJETIVO_OPTUNA = "auc"  # alternativa: "ganancia" con umbral fijo
VERSION_ESTUDIO = "canarios_hist3_rapida_v1"
SEMILLA_BUSQUEDA = 17
SEMILLAS = [17, 103, 211, 307, 401, 503, 601, 701, 809, 907]
N_CONFIGURACIONES_FINAL = 1
CORTES = list(range(10000, 15001, 500))
UMBRAL = 0.025
GANANCIA_ACIERTO = 1_072_500
COSTO_ESTIMULO = 27_500
assert OBJETIVO_OPTUNA in ["auc", "ganancia"]
assert len(SEMILLAS) == len(set(SEMILLAS)) == 10
PARAMETROS_FIJOS = {
    "objective": "binary", "metric": "auc", "max_bin": 31,
    "min_data_in_leaf": 0, "feature_fraction": 0.50,
    "feature_fraction_bynode": 0.20, "learning_rate": 0.005,
    "verbosity": -1, "num_threads": -1,
    "deterministic": True, "force_col_wise": True,
}
# No bagging ni undersampling. Cambia la selección de variables por semilla.
RANGOS = {"num_iterations": [1000, 2000, 100],
          "num_leaves": [8, 128], "min_sum_hessian_in_leaf": [0.1, 100.0]}
# Carpeta exclusiva: evita reutilización de cachés o estudios anteriores.
SALIDA = BASE / "resultados" / ("exploracion19_" + datetime.now().strftime("%Y%m%d_%H%M%S_%f"))
SALIDA.mkdir(parents=True, exist_ok=False)
CSV = SALIDA / "competencia_01_fe_lags_norm.csv"
CORTE_FINAL = 13000
assert CSV_ORIGINAL.is_file(), f"Falta {CSV_ORIGINAL}"

CSV_REFERENCIA = BASE / "referencia" / "SofiaHist3Ensamble_20261010_203913_top12500.csv"


## 1. Reconstruir lags, deltas y normalización desde el original

In [ ]:
from pathlib import Path
import duckdb
import json
from IPython.display import display
BASE = BASE
ENTRADA = BASE / "data" / "competencia_01.csv"
SALIDA_FE = CSV
TRABAJO = SALIDA / "fe"
TRABAJO.mkdir(parents=True, exist_ok=True)
# True: normalizar también lags y deltas. False: normalizar solo originales.
NORMALIZAR_HISTORICOS = True
assert ENTRADA.exists(), f"No existe {ENTRADA}"
assert ENTRADA.resolve() != SALIDA_FE.resolve()
def qi(nombre): return '"' + nombre.replace('"', '""') + '"'
def qs(texto): return "'" + str(texto).replace("'", "''") + "'"
con = duckdb.connect(str(TRABAJO / "features.duckdb"))
con.execute("SET memory_limit='8GB'")
con.execute("SET threads=2")
con.execute("SET preserve_insertion_order=false")
print('DuckDB:', duckdb.__version__)
print('Entrada:', ENTRADA)


con.execute(f"CREATE OR REPLACE TABLE raw AS SELECT * FROM read_csv_auto({qs(ENTRADA)}, sample_size=-1)")
esquema = con.execute("DESCRIBE raw").fetchall()
columnas = [r[0] for r in esquema]
assert {'numero_de_cliente', 'foto_mes', 'clase_ternaria'} <= set(columnas)
assert con.execute('SELECT count(*) FROM raw WHERE numero_de_cliente IS NULL OR foto_mes IS NULL').fetchone()[0] == 0
assert con.execute('SELECT count(*) FROM (SELECT numero_de_cliente, foto_mes FROM raw GROUP BY ALL HAVING count(*)>1)').fetchone()[0] == 0, 'Claves cliente-mes duplicadas'
assert con.execute('SELECT count(*) FROM raw WHERE foto_mes % 100 NOT BETWEEN 1 AND 12').fetchone()[0] == 0
numeric_types = ('TINYINT', 'SMALLINT', 'INTEGER', 'BIGINT', 'HUGEINT', 'UTINYINT', 'USMALLINT', 'UINTEGER', 'UBIGINT', 'FLOAT', 'DOUBLE', 'DECIMAL', 'BOOLEAN')
variables = [nombre for nombre, tipo, *_ in esquema
             if tipo.startswith(numeric_types)
             and nombre not in ('numero_de_cliente', 'foto_mes')
             and not nombre.lower().startswith('clase')]
print('Predictores numéricos:', len(variables))
display(con.execute('SELECT foto_mes, count(*) AS filas FROM raw GROUP BY foto_mes ORDER BY foto_mes').df())


limpias = []
for nombre in columnas:
    q = qi(nombre)
    if nombre in variables:
        expr = f"CASE WHEN isfinite(CAST({q} AS DOUBLE)) THEN CAST({q} AS DOUBLE) ELSE NULL END"
        if nombre == 'ccajas_depositos':
            expr = f"CASE WHEN foto_mes=202105 THEN NULL ELSE {expr} END"
        limpias.append(f"{expr} AS {q}")
    else:
        limpias.append(q)
con.execute('CREATE OR REPLACE TABLE base AS SELECT ' + ', '.join(limpias) +
            ', CAST(floor(foto_mes / 100) AS INTEGER)*12 + CAST(foto_mes % 100 AS INTEGER) AS _mes_idx FROM raw')
if 'ccajas_depositos' in columnas:
    display(con.execute('SELECT foto_mes, count(ccajas_depositos) AS valores_no_nulos FROM base GROUP BY foto_mes ORDER BY foto_mes').df())


con.execute("SET memory_limit='8GB'")
con.execute("SET threads=1")
con.execute("SET preserve_insertion_order=false")

historicas = []
expresiones = []
for v in variables:
    q = qi(v)
    for k in (1, 2):
        lag, delta = f'{v}_lag{k}', f'{v}_delta{k}'
        historicas.extend([lag, delta])
        expresiones.extend([f'p{k}.{q} AS {qi(lag)}',
                            f'b.{q} - p{k}.{q} AS {qi(delta)}'])
assert not set(historicas) & set(columnas), 'El archivo ya tiene nombres de features históricos'
seleccion = ', '.join(['b.' + qi(c) for c in columnas] + expresiones)
con.execute(f"""CREATE OR REPLACE TABLE historia AS
SELECT {seleccion} FROM base b
LEFT JOIN base p1 ON b.numero_de_cliente=p1.numero_de_cliente AND p1._mes_idx=b._mes_idx-1
LEFT JOIN base p2 ON b.numero_de_cliente=p2.numero_de_cliente AND p2._mes_idx=b._mes_idx-2""")
assert con.execute('SELECT count(*) FROM historia').fetchone()[0] == con.execute('SELECT count(*) FROM raw').fetchone()[0]
print('Lags y deltas creados:', len(historicas))


a_normalizar = variables + (historicas if NORMALIZAR_HISTORICOS else [])
normalizadas = [v + '_norm_mes' for v in a_normalizar]
assert not set(normalizadas) & (set(columnas) | set(historicas))
# Cada denominador corresponde a UNA variable y UN mes, no a toda la fila.
maximos = ', '.join(f'max(abs({qi(v)})) AS {qi(v)}' for v in a_normalizar)
con.execute(f'CREATE OR REPLACE TABLE maximos AS SELECT foto_mes, {maximos} FROM historia GROUP BY foto_mes')
expr_norm = []
for v, salida in zip(a_normalizar, normalizadas):
    q = qi(v)
    expr_norm.append(f"CASE WHEN h.{q} IS NULL THEN NULL WHEN m.{q}=0 THEN 0 ELSE h.{q}/m.{q} END AS {qi(salida)}")
con.execute('CREATE OR REPLACE TABLE features AS SELECT h.*, ' + ', '.join(expr_norm) +
            ' FROM historia h JOIN maximos m ON h.foto_mes=m.foto_mes')
print('Originales:', len(columnas), '| Históricas:', len(historicas), '| Normalizadas:', len(normalizadas))
print('Total de columnas:', len(columnas) + len(historicas) + len(normalizadas))


# Controlar que todas las normalizadas estén dentro del rango permitido.
checks = ', '.join(f'count(*) FILTER (WHERE abs({qi(v)}) > 1.000000001) AS {qi(v)}' for v in normalizadas)
fuera_de_rango = con.execute('SELECT ' + checks + ' FROM features').fetchone()
assert sum(fuera_de_rango) == 0
muestra = variables[:2]
campos = ['numero_de_cliente', 'foto_mes']
for v in muestra:
    campos += [v, v+'_lag1', v+'_lag2', v+'_delta1', v+'_delta2', v+'_norm_mes']
display(con.execute('SELECT ' + ', '.join(map(qi, campos)) +
                    ' FROM features ORDER BY numero_de_cliente, foto_mes LIMIT 12').df())
print('Controles correctos. No se reemplazaron lags faltantes por cero.')


SALIDA_FE.parent.mkdir(parents=True, exist_ok=True)
con.execute(f"COPY (SELECT * FROM features ORDER BY numero_de_cliente, foto_mes) TO {qs(SALIDA_FE)} (HEADER, DELIMITER ',')")
manifest = dict(entrada=str(ENTRADA), salida=str(SALIDA_FE),
                duckdb_version=duckdb.__version__, normalizar_historicos=NORMALIZAR_HISTORICOS,
                variables=variables, historicas=historicas, normalizadas=normalizadas,
                correccion='ccajas_depositos=NA en 202105',
                lag='mes calendario exacto', delta='actual menos lag',
                normalizacion='valor / max(abs(valor)) por variable y foto_mes')
(TRABAJO / 'config_fe.json').write_text(json.dumps(manifest, indent=2, ensure_ascii=False), encoding='utf-8')
print('Dataset listo:', SALIDA_FE)
print('Tamaño GB:', round(SALIDA_FE.stat().st_size / 1e9, 3))
con.close()


In [ ]:
assert CSV.is_file(), f"No encuentro {CSV}"
columnas = pd.read_csv(CSV, nrows=0).columns.tolist()
excluir = lambda c: any(c == v or c.startswith(v+'_') for v in ['cprestamos_personales','mprestamos_personales'])
columnas = [c for c in columnas if not excluir(c) and (c.endswith("_norm_mes") or c in ["numero_de_cliente", "foto_mes", "clase_ternaria", "ccajas_depositos"])]
assert any(c.endswith("_norm_mes") for c in columnas), "El CSV debe ser el dataset de FE con columnas _norm_mes"
tipos = {c:'float32' for c in columnas if c not in ['numero_de_cliente','foto_mes','clase_ternaria'] and not c.startswith('clase_')}
partes = []
for chunk in pd.read_csv(CSV, usecols=columnas, dtype=tipos, chunksize=10000, low_memory=True):
    partes.append(chunk.loc[chunk.foto_mes.isin(MESES_FINAL+[MES_PREDICCION])].copy())
data = pd.concat(partes, ignore_index=True)
del partes, chunk
print('Préstamos personales y derivados excluidos en TODAS las etapas.')
CLASES = ["CONTINUA", "BAJA+1", "BAJA+2"]
assert {"numero_de_cliente", "foto_mes", "clase_ternaria", "ccajas_depositos"} <= set(data.columns)
assert not data[["numero_de_cliente", "foto_mes"]].isna().any().any()
data["foto_mes"] = pd.to_numeric(data["foto_mes"], errors="raise").astype(int)
data["clase_ternaria"] = data["clase_ternaria"].astype("string").str.strip()
assert not data.duplicated(["numero_de_cliente", "foto_mes"]).any(), "Duplicados cliente–mes"
assert set(MESES_FINAL + [MES_PREDICCION]).issubset(data.foto_mes.unique())
print("ccajas_depositos de mayo antes de corregir:")
print(data.loc[data.foto_mes == 202105, "ccajas_depositos"].value_counts(dropna=False).head())
data.loc[data.foto_mes == 202105, "ccajas_depositos"] = np.nan
if "ccajas_depositos_norm_mes" in data.columns:
    data.loc[data.foto_mes == 202105, "ccajas_depositos_norm_mes"] = np.nan
# La columna original se leyó solo para auditar calidad; no entra al modelo.
data.drop(columns=["ccajas_depositos"], inplace=True)
features = [c for c in data.columns if c.endswith("_norm_mes")]
assert features and all(not excluir(c) for c in features)
print("Solo predictores normalizados:", len(features))
print("Ejemplos:", features[:12])
no_numericas = data[features].select_dtypes(exclude=["number", "bool"]).columns.tolist()
assert not no_numericas, f"Revisar columnas no numéricas: {no_numericas}"
train = data[data.foto_mes.isin(MESES_BUSQUEDA) & data.clase_ternaria.isin(CLASES)].copy()
valid = data[(data.foto_mes == MES_VALIDACION) & data.clase_ternaria.isin(CLASES)].copy()
X_train = train[features].replace([np.inf, -np.inf], np.nan)
X_valid = valid[features].replace([np.inf, -np.inf], np.nan)
y_train = (train.clase_ternaria == "BAJA+2").astype(int).to_numpy()
y_valid = (valid.clase_ternaria == "BAJA+2").astype(int).to_numpy()
assert len(np.unique(y_train)) == len(np.unique(y_valid)) == 2
assert len(valid) >= max(CORTES)
print("Train:", len(train), "Validación:", len(valid), "Variables:", len(features))
display(pd.crosstab(train.foto_mes, train.clase_ternaria))


## Historia nueva desde el CSV original (incluye julio)
Calcula tendencia y promedio, los normaliza y los agrega al dataset existente.


In [ ]:
import gc
assert CSV_ORIGINAL.is_file(), f"No encuentro {CSV_ORIGINAL}"
# Cálculo vectorizado equivalente a fhistC para ventana de tres filas.
def estadisticas_tres(a, anterior1, anterior2):
    y = np.stack([anterior2, anterior1, a]).astype(np.float64)
    ok = np.isfinite(y)
    x = np.array([1., 2., 3.])[:, None]
    n = ok.sum(axis=0)
    limpio = np.where(ok, y, 0.)
    sy = limpio.sum(axis=0)
    sx = (ok * x).sum(axis=0)
    sxx = (ok * x*x).sum(axis=0)
    sxy = (limpio * x).sum(axis=0)
    denom = n*sxx - sx*sx
    avg = np.full(len(a), np.nan)
    tend = np.full(len(a), np.nan)
    np.divide(sy, n, out=avg, where=n >= 2)
    np.divide(n*sxy - sx*sy, denom, out=tend, where=(n >= 2) & (denom > 0))
    return tend.astype("float32"), avg.astype("float32")

firma_hist = {"version": "ventana3_filas_originales_maxabs_v1",
    "original": str(CSV_ORIGINAL.resolve()), "bytes": CSV_ORIGINAL.stat().st_size,
    "mtime": CSV_ORIGINAL.stat().st_mtime_ns,
    "excluir": ["cprestamos_personales", "mprestamos_personales"]}
clave_hist = hashlib.sha256(json.dumps(firma_hist, sort_keys=True).encode()).hexdigest()[:16]
HIST_DIR = SALIDA / f"historia_{clave_hist}"
HIST_DIR.mkdir(exist_ok=True)
CACHE_HIST = HIST_DIR / "historia3.pkl"
if CACHE_HIST.is_file():
    historia = pd.read_pickle(CACHE_HIST)
    print("Historia recuperada:", CACHE_HIST, flush=True)
else:
    cols_raw = pd.read_csv(CSV_ORIGINAL, nrows=0).columns.tolist()
    meta = ["numero_de_cliente", "foto_mes", "clase_ternaria"]
    cols_raw = [c for c in cols_raw if c in meta or (not excluir(c) and not c.startswith("clase_"))]
    tipos_raw = {c: "float32" for c in cols_raw if c not in meta}
    original = pd.read_csv(CSV_ORIGINAL, usecols=cols_raw, dtype=tipos_raw, low_memory=True)
    assert not original.duplicated(["numero_de_cliente", "foto_mes"]).any()
    original = original.sort_values(["numero_de_cliente", "foto_mes"]).reset_index(drop=True)
    original.loc[original.foto_mes == 202105, "ccajas_depositos"] = np.nan
    cols_hist = [c for c in cols_raw if c not in meta]
    cliente = original.numero_de_cliente.to_numpy()
    meses = original.foto_mes
    iguales1 = np.zeros(len(original), dtype=bool)
    iguales2 = np.zeros(len(original), dtype=bool)
    iguales1[1:] = cliente[1:] == cliente[:-1]
    iguales2[2:] = cliente[2:] == cliente[:-2]
    nuevas = {}
    for i, c in enumerate(cols_hist, 1):
        a = original[c].to_numpy(dtype="float32", copy=True)
        a[~np.isfinite(a)] = np.nan
        a1 = np.roll(a, 1); a1[~iguales1] = np.nan
        a2 = np.roll(a, 2); a2[~iguales2] = np.nan
        tend, avg = estadisticas_tres(a, a1, a2)
        for sufijo, valores in [("tend3", tend), ("avg3", avg)]:
            serie = pd.Series(valores)
            escala = serie.abs().groupby(meses).transform("max").to_numpy()
            norm = np.full(len(serie), np.nan, dtype="float32")
            np.divide(valores, escala, out=norm, where=np.isfinite(escala) & (escala > 0))
            norm[np.isfinite(valores) & (escala == 0)] = 0.
            nuevas[f"{c}_{sufijo}_norm_mes"] = norm
        if i % 25 == 0 or i == len(cols_hist):
            print(f"Historia {i}/{len(cols_hist)}", flush=True)
    historia = pd.concat([original[["numero_de_cliente", "foto_mes"]], pd.DataFrame(nuevas)], axis=1)
    # Julio aporta historia, pero no necesitamos guardar sus filas para entrenar.
    historia = historia[historia.foto_mes.isin(MESES_FINAL + [MES_PREDICCION])].copy()
    temp_hist = CACHE_HIST.with_suffix(".tmp")
    historia.to_pickle(temp_hist)
    temp_hist.replace(CACHE_HIST)
    (HIST_DIR / "configuracion.json").write_text(json.dumps(firma_hist, indent=2), encoding="utf-8")
    del original, nuevas, a, a1, a2, tend, avg
    gc.collect()
cols_nuevas = [c for c in historia.columns if c not in ["numero_de_cliente", "foto_mes"]]
assert not set(cols_nuevas) & set(data.columns), "Ya existen features tend3/avg3"
# No mezclar una fila sin historia con una fila ausente del CSV original.
assert pd.MultiIndex.from_frame(data[["numero_de_cliente", "foto_mes"]]).isin(
    pd.MultiIndex.from_frame(historia[["numero_de_cliente", "foto_mes"]])).all()
data = data.merge(historia, on=["numero_de_cliente", "foto_mes"], how="left", validate="one_to_one")
features += cols_nuevas
train = data[data.foto_mes.isin(MESES_BUSQUEDA) & data.clase_ternaria.isin(CLASES)].copy()
valid = data[(data.foto_mes == MES_VALIDACION) & data.clase_ternaria.isin(CLASES)].copy()
X_train = train[features].replace([np.inf, -np.inf], np.nan)
X_valid = valid[features].replace([np.inf, -np.inf], np.nan)
y_train = (train.clase_ternaria == "BAJA+2").astype(int).to_numpy()
y_valid = (valid.clase_ternaria == "BAJA+2").astype(int).to_numpy()
print("Features históricas nuevas:", len(cols_nuevas), "Total antes de canaritos:", len(features))
del historia
gc.collect()


## 2. Funciones de evaluación
AUC se calcula con rangos promedio para empates, sin importar sklearn. La ganancia por corte usa el ranking descendente y desempata por ID.

In [ ]:
def auc_score(y, p):
    y = np.asarray(y); p = np.asarray(p, dtype=float)
    assert y.shape == p.shape and np.isfinite(p).all()
    n1 = int((y == 1).sum()); n0 = int((y == 0).sum())
    assert n1 > 0 and n0 > 0
    rangos = pd.Series(p).rank(method="average").to_numpy()
    return float((rangos[y == 1].sum() - n1*(n1+1)/2) / (n1*n0))

def ganancia_umbral(y, p):
    pagos = np.where(np.asarray(y) == 1, GANANCIA_ACIERTO, -COSTO_ESTIMULO)
    return int(pagos[np.asarray(p) >= UMBRAL].sum())

def tabla_cortes(y, p, ids):
    ranking = pd.DataFrame({"id": np.asarray(ids), "p": p,
        "pago": np.where(np.asarray(y) == 1, GANANCIA_ACIERTO, -COSTO_ESTIMULO)})
    ranking = ranking.sort_values(["p", "id"], ascending=[False, True])
    acumulada = ranking.pago.cumsum().to_numpy()
    return pd.DataFrame({"envios": CORTES,
                         "ganancia": [int(acumulada[k-1]) for k in CORTES]})

def entrenar(config, dataset, seed, valid_sets=None):
    config = dict(config)
    arboles = int(config.pop("num_iterations"))
    params = {**PARAMETROS_FIJOS, **config, "seed": int(seed),
              "feature_fraction_seed": int(seed), "data_random_seed": int(seed)}
    return lgb.train(params, dataset, num_boost_round=arboles,
        valid_sets=valid_sets, callbacks=[lgb.log_evaluation(0)])


## 3. Selección con canaritos desde cero
Mismos parámetros y criterio de la 13. Sin flags. Se conservan aparte predictores vacíos o constantes.


In [ ]:
import gc
CAN_SEED = 17
CAN_ARBOLES = 1000
CAN_PROPORCION = 0.20
CAN_CONFIG = {"num_iterations": CAN_ARBOLES, "num_leaves": 31,
              "min_sum_hessian_in_leaf": 10.0}
features_antes = list(features)
stat_csv = CSV.stat()
clave_can = {"historia": clave_hist, "csv": str(CSV.resolve()), "bytes": stat_csv.st_size,
    "mtime_ns": stat_csv.st_mtime_ns, "features": features_antes,
    "meses": MESES_BUSQUEDA, "config": CAN_CONFIG, "fijos": PARAMETROS_FIJOS,
    "seed": CAN_SEED, "proporcion": CAN_PROPORCION,
    "regla": "gain_rank_median_minus_sample_std_keep_uninformative_v1",
    "lightgbm": lgb.__version__}
firma_can = hashlib.sha256(json.dumps(clave_can, sort_keys=True).encode()).hexdigest()[:16]
CAN_DIR = SALIDA / f"seleccion_{firma_can}"
CAN_DIR.mkdir(exist_ok=True)
CACHE_CAN = CAN_DIR / "seleccion.json"
if CACHE_CAN.is_file():
    resultado_can = json.loads(CACHE_CAN.read_text(encoding="utf-8"))
    assert resultado_can["protocolo"] == clave_can
    features = resultado_can["features"]
    print("Selección recuperada:", CACHE_CAN, flush=True)
else:
    # No se consideran malas las variables que no tienen variación en el train.
    cardinalidad = X_train.nunique(dropna=True)
    protegidas = cardinalidad[cardinalidad <= 1].index.tolist()
    evaluables = [c for c in features_antes if c not in protegidas]
    assert evaluables, "No hay predictores evaluables"
    n_can = max(10, int(np.ceil(len(evaluables) * CAN_PROPORCION)))
    nombres_can = [f"__canarito_{i:04d}" for i in range(n_can)]
    assert not set(nombres_can) & set(features_antes)
    rng = np.random.default_rng(CAN_SEED)
    matriz_can = X_train[evaluables].copy()
    ruido = pd.DataFrame(rng.random((len(matriz_can), n_can), dtype=np.float32),
                         columns=nombres_can, index=matriz_can.index)
    matriz_can = pd.concat([matriz_can, ruido], axis=1)
    del ruido
    print(f"Seleccionando: {len(evaluables)} variables y {n_can} canaritos; {len(protegidas)} protegidas", flush=True)
    parametros_can = {**PARAMETROS_FIJOS, "num_leaves": 31,
        "min_sum_hessian_in_leaf": 10.0, "seed": CAN_SEED,
        "feature_fraction_seed": CAN_SEED, "data_random_seed": CAN_SEED}
    ds_can = lgb.Dataset(matriz_can, label=y_train)
    modelo_can = lgb.train(parametros_can, ds_can, num_boost_round=CAN_ARBOLES,
                          valid_sets=[ds_can], callbacks=[lgb.log_evaluation(100)])
    importancia = pd.DataFrame({"variable": modelo_can.feature_name(),
         "gain": modelo_can.feature_importance(importance_type="gain")})
    importancia["canarito"] = importancia.variable.isin(nombres_can)
    # Empates conservadores: las variables reales preceden a los canaritos.
    importancia = importancia.sort_values(["gain", "canarito", "variable"],
         ascending=[False, True, True]).reset_index(drop=True)
    importancia["posicion"] = np.arange(1, len(importancia) + 1)
    posiciones = importancia.loc[importancia.canarito, "posicion"]
    limite = float(posiciones.median() - posiciones.std(ddof=1))
    candidatas = set(importancia.loc[(~importancia.canarito) &
                         (importancia.posicion < limite), "variable"])
    candidatas.update(protegidas)
    features = [c for c in features_antes if c in candidatas]
    assert features and any(c in features for c in evaluables), "Filtro sin variables evaluables: revisar importancias"
    importancia["conservada"] = importancia.variable.isin(features)
    importancia.to_csv(CAN_DIR / "importancias.csv", index=False)
    resultado_can = {"protocolo": clave_can, "features": features,
        "protegidas": protegidas, "limite_posicion": limite,
        "cantidad_antes": len(features_antes), "cantidad_despues": len(features)}
    CACHE_CAN.write_text(json.dumps(resultado_can, indent=2), encoding="utf-8")
    del modelo_can, ds_can, matriz_can, importancia
    gc.collect()
print(f"Features conservadas: {len(features)} / {len(features_antes)}", flush=True)
print("Vacías o constantes conservadas aparte:", len(resultado_can["protegidas"]))
print("Selección guardada en:", CAN_DIR)
# Liberar columnas eliminadas antes de optimizar y entrenar modelos finales.
meta_cols = ["numero_de_cliente", "foto_mes", "clase_ternaria"]
data = data[meta_cols + features].copy()
train = train[meta_cols + features].copy()
valid = valid[meta_cols + features].copy()
X_train = train[features].replace([np.inf, -np.inf], np.nan)
X_valid = valid[features].replace([np.inf, -np.inf], np.nan)
gc.collect()
train_lgb = lgb.Dataset(X_train, label=y_train, free_raw_data=False)
valid_lgb = lgb.Dataset(X_valid, label=y_valid, reference=train_lgb, free_raw_data=False)


In [ ]:
FEATURES_REFERENCIA = ['active_quarter_norm_mes', 'internet_norm_mes', 'cliente_edad_norm_mes', 'cliente_antiguedad_norm_mes', 'mrentabilidad_norm_mes', 'mrentabilidad_annual_norm_mes', 'mcomisiones_norm_mes', 'mactivos_margen_norm_mes', 'mpasivos_margen_norm_mes', 'cproductos_norm_mes', 'mcuenta_corriente_norm_mes', 'mcaja_ahorro_norm_mes', 'mcuentas_saldo_norm_mes', 'ctarjeta_debito_transacciones_norm_mes', 'mautoservicio_norm_mes', 'ctarjeta_visa_norm_mes', 'ctarjeta_visa_transacciones_norm_mes', 'mtarjeta_visa_consumo_norm_mes', 'mtarjeta_master_consumo_norm_mes', 'ccaja_seguridad_norm_mes', 'cpayroll_trx_norm_mes', 'mpayroll_norm_mes', 'ccuenta_debitos_automaticos_norm_mes', 'mcuenta_debitos_automaticos_norm_mes', 'cpagomiscuentas_norm_mes', 'mpagomiscuentas_norm_mes', 'ccomisiones_mantenimiento_norm_mes', 'mcomisiones_mantenimiento_norm_mes', 'ccomisiones_otras_norm_mes', 'mcomisiones_otras_norm_mes', 'ctransferencias_recibidas_norm_mes', 'mtransferencias_recibidas_norm_mes', 'ctransferencias_emitidas_norm_mes', 'mtransferencias_emitidas_norm_mes', 'cextraccion_autoservicio_norm_mes', 'mextraccion_autoservicio_norm_mes', 'chomebanking_transacciones_norm_mes', 'ccajas_consultas_norm_mes', 'catm_trx_norm_mes', 'matm_norm_mes', 'ctrx_quarter_norm_mes', 'Master_status_norm_mes', 'Master_mfinanciacion_limite_norm_mes', 'Master_Fvencimiento_norm_mes', 'Master_msaldototal_norm_mes', 'Master_msaldopesos_norm_mes', 'Master_fechaalta_norm_mes', 'Master_mpagominimo_norm_mes', 'Visa_status_norm_mes', 'Visa_Fvencimiento_norm_mes', 'Visa_msaldototal_norm_mes', 'Visa_msaldopesos_norm_mes', 'Visa_mconsumospesos_norm_mes', 'Visa_mconsumosdolares_norm_mes', 'Visa_mlimitecompra_norm_mes', 'Visa_madelantodolares_norm_mes', 'Visa_mpagospesos_norm_mes', 'Visa_fechaalta_norm_mes', 'Visa_mconsumototal_norm_mes', 'Visa_cconsumos_norm_mes', 'Visa_cadelantosefectivo_norm_mes', 'Visa_mpagominimo_norm_mes', 'active_quarter_lag2_norm_mes', 'active_quarter_delta2_norm_mes', 'cliente_vip_lag2_norm_mes', 'cliente_vip_delta2_norm_mes', 'internet_lag2_norm_mes', 'internet_delta2_norm_mes', 'cliente_edad_lag2_norm_mes', 'cliente_edad_delta2_norm_mes', 'cliente_antiguedad_delta1_norm_mes', 'cliente_antiguedad_lag2_norm_mes', 'cliente_antiguedad_delta2_norm_mes', 'mrentabilidad_lag2_norm_mes', 'mrentabilidad_delta2_norm_mes', 'mrentabilidad_annual_lag1_norm_mes', 'mrentabilidad_annual_delta1_norm_mes', 'mrentabilidad_annual_lag2_norm_mes', 'mrentabilidad_annual_delta2_norm_mes', 'mcomisiones_lag1_norm_mes', 'mcomisiones_lag2_norm_mes', 'mcomisiones_delta2_norm_mes', 'mactivos_margen_delta1_norm_mes', 'mactivos_margen_lag2_norm_mes', 'mactivos_margen_delta2_norm_mes', 'mpasivos_margen_lag1_norm_mes', 'mpasivos_margen_lag2_norm_mes', 'mpasivos_margen_delta2_norm_mes', 'cproductos_lag1_norm_mes', 'cproductos_delta1_norm_mes', 'cproductos_lag2_norm_mes', 'cproductos_delta2_norm_mes', 'tcuentas_lag2_norm_mes', 'tcuentas_delta2_norm_mes', 'ccuenta_corriente_delta1_norm_mes', 'ccuenta_corriente_lag2_norm_mes', 'ccuenta_corriente_delta2_norm_mes', 'mcuenta_corriente_adicional_lag2_norm_mes', 'mcuenta_corriente_adicional_delta2_norm_mes', 'mcuenta_corriente_lag1_norm_mes', 'mcuenta_corriente_delta1_norm_mes', 'mcuenta_corriente_lag2_norm_mes', 'mcuenta_corriente_delta2_norm_mes', 'ccaja_ahorro_lag2_norm_mes', 'ccaja_ahorro_delta2_norm_mes', 'mcaja_ahorro_lag1_norm_mes', 'mcaja_ahorro_lag2_norm_mes', 'mcaja_ahorro_delta2_norm_mes', 'mcaja_ahorro_adicional_lag2_norm_mes', 'mcaja_ahorro_adicional_delta2_norm_mes', 'mcaja_ahorro_dolares_lag2_norm_mes', 'mcaja_ahorro_dolares_delta2_norm_mes', 'cdescubierto_preacordado_lag2_norm_mes', 'cdescubierto_preacordado_delta2_norm_mes', 'mcuentas_saldo_lag1_norm_mes', 'mcuentas_saldo_lag2_norm_mes', 'mcuentas_saldo_delta2_norm_mes', 'ctarjeta_debito_lag2_norm_mes', 'ctarjeta_debito_delta2_norm_mes', 'ctarjeta_debito_transacciones_lag2_norm_mes', 'ctarjeta_debito_transacciones_delta2_norm_mes', 'mautoservicio_lag2_norm_mes', 'mautoservicio_delta2_norm_mes', 'ctarjeta_visa_lag2_norm_mes', 'ctarjeta_visa_delta2_norm_mes', 'ctarjeta_visa_transacciones_lag2_norm_mes', 'ctarjeta_visa_transacciones_delta2_norm_mes', 'mtarjeta_visa_consumo_lag2_norm_mes', 'mtarjeta_visa_consumo_delta2_norm_mes', 'ctarjeta_master_lag2_norm_mes', 'ctarjeta_master_delta2_norm_mes', 'ctarjeta_master_transacciones_lag2_norm_mes', 'ctarjeta_master_transacciones_delta2_norm_mes', 'mtarjeta_master_consumo_lag2_norm_mes', 'mtarjeta_master_consumo_delta2_norm_mes', 'cprestamos_prendarios_lag2_norm_mes', 'cprestamos_prendarios_delta2_norm_mes', 'mprestamos_prendarios_lag2_norm_mes', 'mprestamos_prendarios_delta2_norm_mes', 'cprestamos_hipotecarios_lag2_norm_mes', 'cprestamos_hipotecarios_delta2_norm_mes', 'mprestamos_hipotecarios_lag2_norm_mes', 'mprestamos_hipotecarios_delta2_norm_mes', 'cplazo_fijo_lag2_norm_mes', 'cplazo_fijo_delta2_norm_mes', 'mplazo_fijo_dolares_lag2_norm_mes', 'mplazo_fijo_dolares_delta2_norm_mes', 'mplazo_fijo_pesos_lag2_norm_mes', 'mplazo_fijo_pesos_delta2_norm_mes', 'cinversion1_lag2_norm_mes', 'cinversion1_delta2_norm_mes', 'minversion1_pesos_lag2_norm_mes', 'minversion1_pesos_delta2_norm_mes', 'minversion1_dolares_lag2_norm_mes', 'minversion1_dolares_delta2_norm_mes', 'cinversion2_lag2_norm_mes', 'cinversion2_delta2_norm_mes', 'minversion2_lag2_norm_mes', 'minversion2_delta2_norm_mes', 'cseguro_vida_lag2_norm_mes', 'cseguro_vida_delta2_norm_mes', 'cseguro_auto_lag2_norm_mes', 'cseguro_auto_delta2_norm_mes', 'cseguro_vivienda_lag2_norm_mes', 'cseguro_vivienda_delta2_norm_mes', 'cseguro_accidentes_personales_lag2_norm_mes', 'cseguro_accidentes_personales_delta2_norm_mes', 'ccaja_seguridad_lag2_norm_mes', 'ccaja_seguridad_delta2_norm_mes', 'cpayroll_trx_lag1_norm_mes', 'cpayroll_trx_lag2_norm_mes', 'cpayroll_trx_delta2_norm_mes', 'mpayroll_lag1_norm_mes', 'mpayroll_lag2_norm_mes', 'mpayroll_delta2_norm_mes', 'mpayroll2_lag2_norm_mes', 'mpayroll2_delta2_norm_mes', 'cpayroll2_trx_lag2_norm_mes', 'cpayroll2_trx_delta2_norm_mes', 'ccuenta_debitos_automaticos_lag2_norm_mes', 'ccuenta_debitos_automaticos_delta2_norm_mes', 'mcuenta_debitos_automaticos_lag2_norm_mes', 'mcuenta_debitos_automaticos_delta2_norm_mes', 'ctarjeta_visa_debitos_automaticos_lag2_norm_mes', 'ctarjeta_visa_debitos_automaticos_delta2_norm_mes', 'mttarjeta_visa_debitos_automaticos_lag2_norm_mes', 'mttarjeta_visa_debitos_automaticos_delta2_norm_mes', 'ctarjeta_master_debitos_automaticos_lag2_norm_mes', 'ctarjeta_master_debitos_automaticos_delta2_norm_mes', 'mttarjeta_master_debitos_automaticos_lag2_norm_mes', 'mttarjeta_master_debitos_automaticos_delta2_norm_mes', 'cpagodeservicios_lag2_norm_mes', 'cpagodeservicios_delta2_norm_mes', 'mpagodeservicios_lag2_norm_mes', 'mpagodeservicios_delta2_norm_mes', 'cpagomiscuentas_lag2_norm_mes', 'cpagomiscuentas_delta2_norm_mes', 'mpagomiscuentas_lag2_norm_mes', 'mpagomiscuentas_delta2_norm_mes', 'ccajeros_propios_descuentos_lag2_norm_mes', 'ccajeros_propios_descuentos_delta2_norm_mes', 'mcajeros_propios_descuentos_lag2_norm_mes', 'mcajeros_propios_descuentos_delta2_norm_mes', 'ctarjeta_visa_descuentos_lag2_norm_mes', 'ctarjeta_visa_descuentos_delta2_norm_mes', 'mtarjeta_visa_descuentos_lag2_norm_mes', 'mtarjeta_visa_descuentos_delta2_norm_mes', 'ctarjeta_master_descuentos_lag2_norm_mes', 'ctarjeta_master_descuentos_delta2_norm_mes', 'mtarjeta_master_descuentos_lag2_norm_mes', 'mtarjeta_master_descuentos_delta2_norm_mes', 'ccomisiones_mantenimiento_delta1_norm_mes', 'ccomisiones_mantenimiento_lag2_norm_mes', 'ccomisiones_mantenimiento_delta2_norm_mes', 'mcomisiones_mantenimiento_lag2_norm_mes', 'mcomisiones_mantenimiento_delta2_norm_mes', 'ccomisiones_otras_lag2_norm_mes', 'ccomisiones_otras_delta2_norm_mes', 'mcomisiones_otras_lag2_norm_mes', 'mcomisiones_otras_delta2_norm_mes', 'cforex_lag2_norm_mes', 'cforex_delta2_norm_mes', 'cforex_buy_lag2_norm_mes', 'cforex_buy_delta2_norm_mes', 'mforex_buy_lag2_norm_mes', 'mforex_buy_delta2_norm_mes', 'cforex_sell_lag2_norm_mes', 'cforex_sell_delta2_norm_mes', 'mforex_sell_lag2_norm_mes', 'mforex_sell_delta2_norm_mes', 'ctransferencias_recibidas_lag2_norm_mes', 'ctransferencias_recibidas_delta2_norm_mes', 'mtransferencias_recibidas_lag2_norm_mes', 'mtransferencias_recibidas_delta2_norm_mes', 'ctransferencias_emitidas_lag2_norm_mes', 'ctransferencias_emitidas_delta2_norm_mes', 'mtransferencias_emitidas_lag2_norm_mes', 'mtransferencias_emitidas_delta2_norm_mes', 'cextraccion_autoservicio_lag2_norm_mes', 'cextraccion_autoservicio_delta2_norm_mes', 'mextraccion_autoservicio_lag2_norm_mes', 'mextraccion_autoservicio_delta2_norm_mes', 'ccheques_depositados_lag2_norm_mes', 'ccheques_depositados_delta2_norm_mes', 'mcheques_depositados_lag2_norm_mes', 'mcheques_depositados_delta2_norm_mes', 'ccheques_emitidos_lag2_norm_mes', 'ccheques_emitidos_delta2_norm_mes', 'mcheques_emitidos_lag2_norm_mes', 'mcheques_emitidos_delta2_norm_mes', 'ccheques_depositados_rechazados_lag2_norm_mes', 'ccheques_depositados_rechazados_delta2_norm_mes', 'mcheques_depositados_rechazados_lag2_norm_mes', 'mcheques_depositados_rechazados_delta2_norm_mes', 'ccheques_emitidos_rechazados_lag2_norm_mes', 'ccheques_emitidos_rechazados_delta2_norm_mes', 'mcheques_emitidos_rechazados_lag2_norm_mes', 'mcheques_emitidos_rechazados_delta2_norm_mes', 'tcallcenter_lag2_norm_mes', 'tcallcenter_delta2_norm_mes', 'ccallcenter_transacciones_lag2_norm_mes', 'ccallcenter_transacciones_delta2_norm_mes', 'thomebanking_lag2_norm_mes', 'thomebanking_delta2_norm_mes', 'chomebanking_transacciones_lag2_norm_mes', 'chomebanking_transacciones_delta2_norm_mes', 'ccajas_transacciones_lag2_norm_mes', 'ccajas_transacciones_delta2_norm_mes', 'ccajas_consultas_lag2_norm_mes', 'ccajas_consultas_delta2_norm_mes', 'ccajas_depositos_lag2_norm_mes', 'ccajas_depositos_delta2_norm_mes', 'ccajas_extracciones_lag2_norm_mes', 'ccajas_extracciones_delta2_norm_mes', 'ccajas_otras_lag2_norm_mes', 'ccajas_otras_delta2_norm_mes', 'catm_trx_lag2_norm_mes', 'catm_trx_delta2_norm_mes', 'matm_lag2_norm_mes', 'matm_delta2_norm_mes', 'catm_trx_other_lag2_norm_mes', 'catm_trx_other_delta2_norm_mes', 'matm_other_lag2_norm_mes', 'matm_other_delta2_norm_mes', 'ctrx_quarter_lag1_norm_mes', 'ctrx_quarter_delta1_norm_mes', 'ctrx_quarter_lag2_norm_mes', 'ctrx_quarter_delta2_norm_mes', 'tmobile_app_lag2_norm_mes', 'tmobile_app_delta2_norm_mes', 'cmobile_app_trx_lag2_norm_mes', 'cmobile_app_trx_delta2_norm_mes', 'Master_delinquency_lag2_norm_mes', 'Master_delinquency_delta2_norm_mes', 'Master_status_lag2_norm_mes', 'Master_status_delta2_norm_mes', 'Master_mfinanciacion_limite_lag2_norm_mes', 'Master_mfinanciacion_limite_delta2_norm_mes', 'Master_Fvencimiento_lag2_norm_mes', 'Master_Fvencimiento_delta2_norm_mes', 'Master_Finiciomora_lag2_norm_mes', 'Master_Finiciomora_delta2_norm_mes', 'Master_msaldototal_lag2_norm_mes', 'Master_msaldototal_delta2_norm_mes', 'Master_msaldopesos_lag2_norm_mes', 'Master_msaldopesos_delta2_norm_mes', 'Master_msaldodolares_lag2_norm_mes', 'Master_msaldodolares_delta2_norm_mes', 'Master_mconsumospesos_lag2_norm_mes', 'Master_mconsumospesos_delta2_norm_mes', 'Master_mconsumosdolares_lag2_norm_mes', 'Master_mconsumosdolares_delta2_norm_mes', 'Master_mlimitecompra_lag2_norm_mes', 'Master_mlimitecompra_delta2_norm_mes', 'Master_madelantopesos_lag2_norm_mes', 'Master_madelantopesos_delta2_norm_mes', 'Master_madelantodolares_lag2_norm_mes', 'Master_madelantodolares_delta2_norm_mes', 'Master_fultimo_cierre_lag2_norm_mes', 'Master_fultimo_cierre_delta2_norm_mes', 'Master_mpagado_lag2_norm_mes', 'Master_mpagado_delta2_norm_mes', 'Master_mpagospesos_lag2_norm_mes', 'Master_mpagospesos_delta2_norm_mes', 'Master_mpagosdolares_lag2_norm_mes', 'Master_mpagosdolares_delta2_norm_mes', 'Master_fechaalta_lag2_norm_mes', 'Master_fechaalta_delta2_norm_mes', 'Master_mconsumototal_lag2_norm_mes', 'Master_mconsumototal_delta2_norm_mes', 'Master_cconsumos_lag2_norm_mes', 'Master_cconsumos_delta2_norm_mes', 'Master_cadelantosefectivo_lag2_norm_mes', 'Master_cadelantosefectivo_delta2_norm_mes', 'Master_mpagominimo_lag2_norm_mes', 'Master_mpagominimo_delta2_norm_mes', 'Visa_delinquency_lag2_norm_mes', 'Visa_delinquency_delta2_norm_mes', 'Visa_status_lag2_norm_mes', 'Visa_status_delta2_norm_mes', 'Visa_mfinanciacion_limite_lag2_norm_mes', 'Visa_mfinanciacion_limite_delta2_norm_mes', 'Visa_Fvencimiento_lag2_norm_mes', 'Visa_Fvencimiento_delta2_norm_mes', 'Visa_Finiciomora_lag2_norm_mes', 'Visa_Finiciomora_delta2_norm_mes', 'Visa_msaldototal_lag2_norm_mes', 'Visa_msaldototal_delta2_norm_mes', 'Visa_msaldopesos_lag2_norm_mes', 'Visa_msaldopesos_delta2_norm_mes', 'Visa_msaldodolares_lag2_norm_mes', 'Visa_msaldodolares_delta2_norm_mes', 'Visa_mconsumospesos_lag2_norm_mes', 'Visa_mconsumospesos_delta2_norm_mes', 'Visa_mconsumosdolares_lag2_norm_mes', 'Visa_mconsumosdolares_delta2_norm_mes', 'Visa_mlimitecompra_lag2_norm_mes', 'Visa_mlimitecompra_delta2_norm_mes', 'Visa_madelantopesos_lag2_norm_mes', 'Visa_madelantopesos_delta2_norm_mes', 'Visa_madelantodolares_lag2_norm_mes', 'Visa_madelantodolares_delta2_norm_mes', 'Visa_fultimo_cierre_lag2_norm_mes', 'Visa_fultimo_cierre_delta2_norm_mes', 'Visa_mpagado_lag2_norm_mes', 'Visa_mpagado_delta2_norm_mes', 'Visa_mpagospesos_lag2_norm_mes', 'Visa_mpagospesos_delta2_norm_mes', 'Visa_mpagosdolares_lag2_norm_mes', 'Visa_mpagosdolares_delta2_norm_mes', 'Visa_fechaalta_lag2_norm_mes', 'Visa_fechaalta_delta2_norm_mes', 'Visa_mconsumototal_lag2_norm_mes', 'Visa_mconsumototal_delta2_norm_mes', 'Visa_cconsumos_lag2_norm_mes', 'Visa_cconsumos_delta2_norm_mes', 'Visa_cadelantosefectivo_lag2_norm_mes', 'Visa_cadelantosefectivo_delta2_norm_mes', 'Visa_mpagominimo_lag2_norm_mes', 'Visa_mpagominimo_delta2_norm_mes', 'cliente_antiguedad_tend3_norm_mes', 'mrentabilidad_annual_tend3_norm_mes', 'mrentabilidad_annual_avg3_norm_mes', 'mactivos_margen_tend3_norm_mes', 'mactivos_margen_avg3_norm_mes', 'mpasivos_margen_avg3_norm_mes', 'cproductos_tend3_norm_mes', 'cproductos_avg3_norm_mes', 'ccuenta_corriente_tend3_norm_mes', 'mcuenta_corriente_tend3_norm_mes', 'mcuenta_corriente_avg3_norm_mes', 'mcaja_ahorro_avg3_norm_mes', 'mcuentas_saldo_avg3_norm_mes', 'mtarjeta_visa_consumo_avg3_norm_mes', 'cpayroll_trx_avg3_norm_mes', 'mpayroll_avg3_norm_mes', 'mcomisiones_mantenimiento_avg3_norm_mes', 'mcomisiones_otras_avg3_norm_mes', 'ctransferencias_recibidas_avg3_norm_mes', 'ctrx_quarter_avg3_norm_mes']
seleccion_igual = features == FEATURES_REFERENCIA
reporte_seleccion = {"coincide_lista_y_orden": seleccion_igual,
    "cantidad": len(features), "esperada": len(FEATURES_REFERENCIA),
    "faltan": sorted(set(FEATURES_REFERENCIA)-set(features)),
    "nuevas": sorted(set(features)-set(FEATURES_REFERENCIA))}
(SALIDA / "comparacion_seleccion.json").write_text(json.dumps(reporte_seleccion, indent=2), encoding="utf-8")
print("Selección idéntica a las 400 variables anteriores:", seleccion_igual)
if not seleccion_igual:
    print("ATENCIÓN: cambió la selección; esta corrida no reproduce la configuración original. No se reemplaza la selección calculada.")


## 4. Repetir los cinco trials originales
Estudio vacío, semilla 17, AUC de junio. Rango de árboles: 1000–2000, paso 100. No se fijan los parámetros elegidos.


In [ ]:
def objective(trial):
    config = {
        "num_iterations": trial.suggest_int("num_iterations", *RANGOS["num_iterations"][:2], step=RANGOS["num_iterations"][2]),
        "num_leaves": trial.suggest_int("num_leaves", *RANGOS["num_leaves"]),
        "min_sum_hessian_in_leaf": trial.suggest_float("min_sum_hessian_in_leaf", *RANGOS["min_sum_hessian_in_leaf"], log=True),
    }
    model = entrenar(config, train_lgb, SEMILLA_BUSQUEDA, [valid_lgb])
    p = model.predict(X_valid)
    auc = auc_score(y_valid, p)
    gan = ganancia_umbral(y_valid, p)
    trial.set_user_attr("auc_junio", auc)
    trial.set_user_attr("ganancia_umbral_junio", gan)
    return auc if OBJETIVO_OPTUNA == "auc" else gan

h = hashlib.sha256()
with CSV.open("rb") as f:
    for b in iter(lambda: f.read(1024*1024), b""):
        h.update(b)
HASH_CSV = h.hexdigest()
protocolo = {"version": VERSION_ESTUDIO, "train": MESES_BUSQUEDA, "valid": MES_VALIDACION,
    "objetivo": OBJETIVO_OPTUNA, "fijos": PARAMETROS_FIJOS, "rangos": RANGOS,
    "seed": SEMILLA_BUSQUEDA, "features": features, "firma_canaritos": firma_can, "calidad": "ccajas_depositos_mayo_nan",
    "lgb": lgb.__version__, "optuna": optuna.__version__, "umbral": UMBRAL,
    "ganancia_acierto": GANANCIA_ACIERTO, "costo": COSTO_ESTIMULO}
firma = hashlib.sha256(json.dumps(protocolo, sort_keys=True).encode()).hexdigest()[:12]
estudio = f"profesor_{HASH_CSV[:12]}_{firma}"
RUN_DIR = SALIDA / estudio
RUN_DIR.mkdir(exist_ok=True)
study = optuna.create_study(direction="maximize", study_name=estudio,
    sampler=optuna.samplers.TPESampler(seed=SEMILLA_BUSQUEDA))
study.optimize(objective, n_trials=N_TRIALS, n_jobs=1)
completos = [t for t in study.trials if t.state == optuna.trial.TrialState.COMPLETE]
assert len(completos) >= N_CONFIGURACIONES_FINAL, "Completar al menos diez trials."
print("Mejor trial:", study.best_trial.number, "Objetivo:", OBJETIVO_OPTUNA, "Valor:", study.best_value)
display(study.trials_dataframe().sort_values("value", ascending=False).head(10))
study.trials_dataframe().to_csv(RUN_DIR / "trials.csv", index=False)


In [ ]:
PARAMETROS_REFERENCIA = {"num_iterations": 1500, "num_leaves": 103,
                         "min_sum_hessian_in_leaf": 9.311090875055461}
print("Parámetros encontrados:", study.best_params)
print("Coinciden con ganador original:", study.best_params == PARAMETROS_REFERENCIA)
print("AUC original de referencia: 0.9048495713188451")


## 5. Validación del ensamble elegido en junio

In [ ]:
config_mejor = dict(study.best_params)
probs = []
filas = []
filas_cortes = []
for i, seed in enumerate(SEMILLAS, 1):
    modelo = entrenar(config_mejor, train_lgb, seed)
    p = modelo.predict(X_valid)
    probs.append(p)
    filas.append({"seed": seed, "auc": auc_score(y_valid, p),
                  "ganancia_umbral": ganancia_umbral(y_valid, p)})
    tc = tabla_cortes(y_valid, p, valid.numero_de_cliente)
    tc["seed"] = seed
    filas_cortes.append(tc)
    print(f"Semilla {i}/{len(SEMILLAS)} terminada", flush=True)
por_semilla = pd.DataFrame(filas)
por_corte = pd.concat(filas_cortes, ignore_index=True)
print("Media y desvío de resultados individuales:")
display(por_semilla[["auc", "ganancia_umbral"]].agg(["mean", "std", "min", "max"]))
resumen_cortes = por_corte.groupby("envios").ganancia.agg(["mean", "std", "min", "max"])
display(resumen_cortes)
por_semilla.to_csv(RUN_DIR / "validacion_3_semillas.csv", index=False)
por_corte.to_csv(RUN_DIR / "cortes_3_semillas.csv", index=False)

p_valid = np.mean(probs, axis=0)
print("AUC ensamble junio:", auc_score(y_valid, p_valid))
tabla_cortes(y_valid, p_valid, valid.numero_de_cliente).to_csv(RUN_DIR / "validacion_ensamble.csv", index=False)
del probs


In [ ]:
configs_finales = [{"trial": study.best_trial.number, "params": dict(config_mejor), "seed": seed} for seed in SEMILLAS]
versiones = {p: metadata.version(p) for p in ["pandas", "numpy", "lightgbm", "optuna", "duckdb", "ipykernel"]}
configuracion = {"protocolo": protocolo, "sha256_csv": HASH_CSV, "historia": firma_hist,
    "semillas": SEMILLAS, "mejor_configuracion": config_mejor,
    "configuraciones_finales": configs_finales,
    "meses_final": MESES_FINAL, "mes_prediccion": MES_PREDICCION,
    "version_python": platform.python_version(), "versiones": versiones}
(RUN_DIR / "configuracion.json").write_text(json.dumps(configuracion, indent=2), encoding="utf-8")
(RUN_DIR / "requirements_versiones.txt").write_text("\n".join(f"{p}=={v}" for p,v in versiones.items()), encoding="utf-8")

configuracion["n_trials"] = N_TRIALS
configuracion["flags_zero"] = []
configuracion["corte_final"] = CORTE_FINAL


## 6. Entrenamiento final y predicción de agosto

In [ ]:
final = data[data.foto_mes.isin(MESES_FINAL) & data.clase_ternaria.isin(CLASES)].copy()
agosto = data[data.foto_mes == MES_PREDICCION].copy()
assert len(agosto) >= max(CORTES)
assert agosto.numero_de_cliente.notna().all() and agosto.numero_de_cliente.is_unique
X_final = final[features].replace([np.inf, -np.inf], np.nan)
y_final = (final.clase_ternaria == "BAJA+2").astype(int).to_numpy()
X_agosto = agosto[features].replace([np.inf, -np.inf], np.nan)
ds_final = lgb.Dataset(X_final, label=y_final, free_raw_data=False)
firma_final = hashlib.sha256(json.dumps(configs_finales, sort_keys=True).encode()).hexdigest()[:8]
nombre = f"SofiaCerca19_{firma}_{firma_final}"
ENVIO_DIR = SALIDA / "envios" / nombre
ENVIO_DIR.mkdir(parents=True, exist_ok=True)
archivos = []
ids = pd.to_numeric(agosto.numero_de_cliente, errors="raise")
assert np.isfinite(ids.to_numpy(dtype=float)).all() and (ids == np.floor(ids)).all()
for i, conf in enumerate(configs_finales, 1):
    seed = conf["seed"]
    ruta_modelo = ENVIO_DIR / f"modelo_seed{seed}.txt"
    m = entrenar(conf["params"], ds_final, seed)
    m.save_model(str(ruta_modelo))
    p = m.predict(X_agosto)
    ranking = pd.DataFrame({"numero_de_cliente": ids.astype("int64").to_numpy(), "probabilidad": p})
    ranking = ranking.sort_values(["probabilidad", "numero_de_cliente"], ascending=[False, True])
    ranking.to_csv(ENVIO_DIR / f"ranking_seed{seed}_ANALISIS.csv", index=False)
    for k in CORTES:
        destino = ENVIO_DIR / f"{nombre}_seed{seed}_top{k}.csv"
        ranking.head(k)[["numero_de_cliente"]].to_csv(destino, index=False, header=False)
        lineas = destino.read_text().splitlines()
        assert len(lineas) == k and len(set(lineas)) == k and all(s.isdigit() for s in lineas)
        archivos.append({"seed": seed, "envios": k, "archivo": destino.name})
    print(f"Modelo final {i}/{len(configs_finales)} listo, semilla {seed}", flush=True)
configuracion["archivos_candidatos"] = archivos
(ENVIO_DIR / "configuracion.json").write_text(json.dumps(configuracion, indent=2), encoding="utf-8")
(ENVIO_DIR / "requirements_versiones.txt").write_text((RUN_DIR / "requirements_versiones.txt").read_text(), encoding="utf-8")
print("Carpeta:", ENVIO_DIR)
print("Adjuntar solo CSV con seed y top en el nombre; los rankings son para análisis.")


## 7. Ensamble y candidato de 13.000 clientes

In [ ]:
import re

CORTES_ENSAMBLE = CORTES

# Rankings individuales del experimento actual.
rutas = [ENVIO_DIR / f"ranking_seed{seed}_ANALISIS.csv" for seed in SEMILLAS]
assert all(p.is_file() for p in rutas)
assert rutas, f"No encontré rankings en {ENVIO_DIR}"

por_seed = {}

for ruta in rutas:
    coincidencia = re.fullmatch(
        r"ranking_seed(\d+)_ANALISIS\.csv", ruta.name
    )
    if coincidencia:
        seed = int(coincidencia.group(1))
        por_seed.setdefault(seed, []).append(ruta)

suma = None
semillas_usadas = []

for seed, archivos in sorted(por_seed.items()):
    serie_seed = None

    for ruta in archivos:
        df = pd.read_csv(
            ruta,
            dtype={"numero_de_cliente": "int64"}
        )
        assert {"numero_de_cliente", "probabilidad"} <= set(df.columns)
        assert df.numero_de_cliente.is_unique
        assert np.isfinite(df.probabilidad).all()
        assert df.probabilidad.between(0, 1).all()

        serie = df.set_index("numero_de_cliente")[
            "probabilidad"
        ].sort_index()

        # Si una semilla aparece en dos carpetas, debe ser el mismo ranking.
        if serie_seed is None:
            serie_seed = serie
        else:
            assert serie.index.equals(serie_seed.index), (
                f"IDs distintos para semilla {seed}"
            )
            assert np.allclose(
                serie.to_numpy(), serie_seed.to_numpy(),
                rtol=1e-10, atol=1e-12
            ), f"Hay predicciones distintas para la semilla {seed}"

    if suma is None:
        suma = serie_seed.copy()
    else:
        assert suma.index.equals(serie_seed.index), (
            f"La semilla {seed} tiene otros clientes"
        )
        suma += serie_seed

    semillas_usadas.append(seed)

assert len(semillas_usadas) >= 2, "Se necesitan al menos dos modelos"

# Comprobar que son todos los clientes de agosto de esta notebook.
ids_agosto = pd.to_numeric(
    agosto.numero_de_cliente, errors="raise"
)
assert ids_agosto.notna().all()
assert (ids_agosto == np.floor(ids_agosto)).all()
assert set(suma.index) == set(ids_agosto.astype("int64"))

promedio = suma / len(semillas_usadas)

ranking_ensamble = (
    promedio.rename("probabilidad")
    .rename_axis("numero_de_cliente")
    .reset_index()
    .sort_values(
        ["probabilidad", "numero_de_cliente"],
        ascending=[False, True]
    )
)

# Carpeta nueva para identificar exactamente este ensamble.
marca = datetime.now().strftime("%Y%m%d_%H%M%S")
prefijo = f"SofiaCerca19Ensamble_{marca}"
CARPETA_ENSAMBLE = ENVIO_DIR / prefijo
CARPETA_ENSAMBLE.mkdir(exist_ok=False)

ranking_ensamble.to_csv(
    CARPETA_ENSAMBLE / "ranking_ensamble_ANALISIS.csv",
    index=False
)

resumen = {
    "metodo": "promedio simple de probabilidades",
    "semillas": semillas_usadas,
    "cantidad_modelos": len(semillas_usadas),
    "cortes": CORTES_ENSAMBLE,
    "archivos_fuente": [
        str(ruta)
        for seed in semillas_usadas
        for ruta in por_seed[seed]
    ],
}
(CARPETA_ENSAMBLE / "ensamble.json").write_text(
    json.dumps(resumen, indent=2), encoding="utf-8"
)

print("Semillas incluidas:", semillas_usadas)
print("Cantidad de modelos:", len(semillas_usadas))

for corte in CORTES_ENSAMBLE:
    assert 8000 <= corte <= 15000
    assert corte <= len(ranking_ensamble)

    destino = CARPETA_ENSAMBLE / f"{prefijo}_top{corte}.csv"
    ranking_ensamble.head(corte)[["numero_de_cliente"]].to_csv(
        destino, index=False, header=False
    )

    lineas = destino.read_text().splitlines()
    assert len(lineas) == corte
    assert len(set(lineas)) == corte
    assert all(linea.isdigit() for linea in lineas)

    print("CSV listo:", destino.name)

print("\nCarpeta:", CARPETA_ENSAMBLE)
print("Adjuntar únicamente los archivos *_top*.csv")
assert semillas_usadas == sorted(SEMILLAS)
CSV_FINAL = CARPETA_ENSAMBLE / f"{prefijo}_top{CORTE_FINAL}.csv"
configuracion["csv_final"] = CSV_FINAL.name
configuracion["sha256_csv_final"] = hashlib.sha256(CSV_FINAL.read_bytes()).hexdigest()
(CARPETA_ENSAMBLE / "configuracion.json").write_text(json.dumps(configuracion, indent=2), encoding="utf-8")
print("Candidato final:", CSV_FINAL)


La notebook fue revisada sintácticamente. Falta entrenar con el dataset y comparar dos ejecuciones completas para comprobar la reproducción. Guardar requirements_versiones.txt, trials.csv, seleccion.json, importancias.csv, configuracion.json y el CSV elegido. Instalar las versiones registradas antes de repetir. No modificar N_TRIALS ni SEMILLAS después de elegir el envío sin generar y probar un nuevo candidato.

In [ ]:
if CSV_REFERENCIA.is_file():
    original_ids = pd.read_csv(CSV_REFERENCIA, header=None, names=["id"], dtype={"id":"int64"})
    nuevos_ids = pd.read_csv(CSV_FINAL, header=None, names=["id"], dtype={"id":"int64"})
    assert original_ids.id.is_unique and len(original_ids) == CORTE_FINAL
    reporte = {"mismos_clientes": set(original_ids.id) == set(nuevos_ids.id),
               "mismo_orden": original_ids.id.tolist() == nuevos_ids.id.tolist(),
               "mismos_bytes": CSV_REFERENCIA.read_bytes() == CSV_FINAL.read_bytes(),
               "clientes_comunes": len(set(original_ids.id) & set(nuevos_ids.id)),
               "sha256_referencia": hashlib.sha256(CSV_REFERENCIA.read_bytes()).hexdigest(),
               "sha256_nuevo": hashlib.sha256(CSV_FINAL.read_bytes()).hexdigest()}
    (CARPETA_ENSAMBLE / "comparacion_csv.json").write_text(json.dumps(reporte, indent=2), encoding="utf-8")
    print(reporte)
else:
    print("No se comparó con el ganador: falta", CSV_REFERENCIA)
